In [72]:
# add filter
import copy
import glob
import numpy as np
from datetime import datetime, timedelta
import os
import re
import requests
import json
from astropy.time import Time
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

def standardize_to_degrees(ra, dec):
    ra_str = str(ra).strip()
    
    if any(char in ra_str for char in [':', 'h', ' ']):
        # It is sexagesimal. We assume RA is in hours and Dec is in degrees.
        coord = SkyCoord(ra=ra, dec=dec, unit=(u.hourangle, u.deg))
        return coord.ra.deg, coord.dec.deg
    else:
        return float(ra), float(dec)
        

In [73]:
# get your LCO token from https://observe.lco.global/accounts/profile

from dotenv import load_dotenv

load_dotenv()

LCO_TOKEN = os.getenv('LCO_TOKEN')

if LCO_TOKEN:
    print("Successfully loaded the LCO token!")
else:
    print("Token not found. Check your .env file.")
    
requestpath  = "https://observe.lco.global/api/requestgroups/"

Successfully loaded the LCO token!


In [108]:
# path to your usual DECam script
date = 'June2'
decam_script_path = glob.glob(f'../{date}/scriptsytabledeoffsets/*.json')


for decam_json_file in decam_script_path:
    decam_script = pd.read_json(decam_json_file)


    example_path = '../example.json'
    
    with open(example_path, "r") as f:
            data = json.load(f)
    
    i = 0
    
    # load name, ra, dec, propid
    name = decam_script['object'].iloc[i]
    ra = decam_script['RA'].iloc[i]
    dec = decam_script['dec'].iloc[i]
    ra, dec = standardize_to_degrees(ra, dec)
    # propid = decam_script['propid'].iloc[i]
    propid = '2013A-9999'
    
    # modify priority
    data['proposal'] = propid
    data['ipp_value'] = 0.9
    data
    
    # modify target
    target = data['requests'][0]['configurations'][0]['target']
    target['name'] = name
    data['name'] = name
    
    target['ra'] = ra
    target['dec'] = dec
    data['requests'][0]['configurations'][0]['extra_params']['detector_centering'] = 'central_gap'
    # print(data['requests'][0]['configurations'][0]['instrument_configs'])
    
    for i in range(len(decam_script)):
        # configurations : filters and offsets
        # load filters and offsets
        
        filt = decam_script['filter'].iloc[i]
        exp_time = decam_script['expTime'].iloc[i]
        deltaRA = decam_script['deltaRA'].iloc[i]
        deltaRA = 0.0 if pd.isna(deltaRA) else float(deltaRA)
        deltaDEC = decam_script['deltaDEC'].iloc[i]
        deltaDEC = 0.0 if pd.isna(deltaDEC) else float(deltaDEC)
        
        # print(deltaRA, deltaDEC)
        if i == 0:
    
            instrument_configs = data['requests'][0]['configurations'][0]['instrument_configs']
            instrument_configs[0]['exposure_time'] = int(exp_time)
            instrument_configs[0]['optical_elements']['filter'] = filt
            instrument_configs[0]['extra_params']['offset_ra'] = deltaRA
            instrument_configs[0]['extra_params']['offset_dec'] = deltaDEC
            print(deltaRA, instrument_configs[0]['extra_params']['offset_ra'])
    
            original_config = instrument_configs[0]
            print(original_config['extra_params']['offset_ra'])
        
        if i > 0:
            added_config = copy.deepcopy(original_config)
            added_config['exposure_time'] = int(exp_time)
            added_config['optical_elements']['filter'] = filt
            added_config['extra_params']['offset_ra'] = deltaRA
            added_config['extra_params']['offset_dec'] = deltaDEC
            instrument_configs.append(added_config)
        
        print(instrument_configs[i]['extra_params']['offset_ra'])
    # time window
    window_date = Time('2026-06-02 00:00:00',format='iso')
    window_original = data['requests'][0]['windows'][0]
    window_original['start'] = (window_date - timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M:%S")
    window_original['end'] = (window_date + timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
    
    
    # posting
    
    response = requests.post(
                requestpath,
                headers={"Authorization": f"Token {LCO_TOKEN}"},
                json=data,  # Make sure you use json!
            )
    
    
    sent,failed = [],[]
    
    if response.status_code == 400:
        print('Failed to send request, it is not in the queue')
        failed.append([data['name'],response.text])
    if response.status_code == 200 or response.status_code == 201:
        print(f'Success!! request should be in the queue. https://observe.lco.global/requests/{response.json()['requests'][0]['id']}')
        print('name:', data['name'], 'RequestGroup:',response.json()['id'])
        sent.append(response.json()['id'])
        
    # data
    np.save(f'../{date}/{decam_json_file}_sent.npy',sent)
    np.save(f'../{date}/{decam_json_file}_failed.npy',failed)

0.0 0.0
0.0
0.0
100.0
-199.0
-94.0
460.0
150.0
-184.0
707.0
-1886.0
2371.0
-1420.0
226.0
-1995.0
1253.0
881.0
-3000.0
363.0
1000.0
-2518.0
5404.0
1419.0
-3028.0
Success!! request should be in the queue. https://observe.lco.global/requests/4207661
name: star flat z RequestGroup: 2526931
0.0 0.0
0.0
0.0
100.0
-199.0
-94.0
460.0
150.0
-184.0
707.0
-1886.0
2371.0
-1420.0
226.0
-1995.0
1253.0
881.0
-3000.0
363.0
1000.0
-2518.0
5404.0
1419.0
-3028.0
Success!! request should be in the queue. https://observe.lco.global/requests/4207662
name: star flat i RequestGroup: 2526932
0.0 0.0
0.0
0.0
100.0
-199.0
-94.0
460.0
150.0
-184.0
707.0
-1886.0
2371.0
-1420.0
226.0
-1995.0
1253.0
881.0
-3000.0
363.0
1000.0
-2518.0
5404.0
1419.0
-3028.0
Success!! request should be in the queue. https://observe.lco.global/requests/4207663
name: star flat g RequestGroup: 2526933
0.0 0.0
0.0
0.0
100.0
-199.0
-94.0
460.0
150.0
-184.0
707.0
-1886.0
2371.0
-1420.0
226.0
-1995.0
1253.0
881.0
-3000.0
363.0
1000.0
-2518.0


In [107]:
data

{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.6,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {}},
     'instrument_configs': [{'exposure_time': 30,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0.0, 'offset_dec': 0.0}},
      {'exposure_time': 30,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 100.0, 'offset_dec': 65.0}},
      {'exposure_time': 30,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': -199.0, 'offset_dec': -8.0}},
     

In [67]:
data

{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.6,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {}},
     'instrument_configs': [{'exposure_time': 30,
       'optical_elements': {'filter': 'g'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': -3028.0, 'offset_dec': -1857.0}},
      {'exposure_time': 30,
       'optical_elements': {'filter': 'g'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 100.0, 'offset_dec': 65.0}},
      {'exposure_time': 30,
       'optical_elements': {'filter': 'g'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': -199.0, 'offset_dec': -8.0}